# 02 · Equal-budget hyper-parameter tuning
Every model gets the **same budget**: `trials` Optuna (TPE) trials on S-C **validation** macro-F1 at window length `tune_L` (seed 0). The default configuration is always trial 0. The test set is never touched here.

Output: `results/<dataset>/best_hp.json` and every trial in `tuning_log.jsonl` (Table 2 of the paper).

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 60)
from sdnids_bench.config import load_config, mode_params, active_datasets, enable_determinism
cfg = load_config('..')
mp = mode_params(cfg)
enable_determinism()
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
from sdnids_bench.experiments import tune, ds_dir
DATASETS = sorted(set(sum([active_datasets(cfg, e) for e in cfg['experiments']], [])), key=list(cfg['datasets']).index)
print('tuning on:', DATASETS, '| trials per model:', mp['trials'])

In [ ]:
tune(cfg, DATASETS, cfg['tasks'], cfg['models'], L=cfg['tune_L'], trials=mp['trials'])

In [ ]:
for ds in DATASETS:
    log = pd.read_json(os.path.join(ds_dir(cfg, ds), 'tuning_log.jsonl'), lines=True)
    s = log.groupby(['task','model']).agg(trials=('trial','count'), best_val_macro_f1=('val_macro_f1','max'), minutes=('wall_s', lambda x: x.sum()/60))
    print(ds); display(s.round(4))